## Author: Enrique Posada

In [1]:
import requests
from sklearn.metrics import classification_report, confusion_matrix


In [2]:
# --- Test Dataset ---
test_data = [
    ("I love this movie, it was amazing and fun!", "Positive"),
    ("This was the worst experience I’ve ever had.", "Negative"),
    ("The product is okay, nothing special.", "Neutral"),
    ("Absolutely fantastic performance!", "Positive"),
    ("I am very disappointed with the service.", "Negative"),
    ("It works as expected.", "Neutral"),
    ("The food was delicious and I would come again.", "Positive"),
    ("I hate how slow this app is.", "Negative"),
    ("It is fine, neither good nor bad.", "Neutral"),
    ("What a wonderful surprise, I really enjoyed it!", "Positive"),
]

In [3]:
# --- Sentiment Analysis Function ---
def sentiment_analysis(user_query):
    """Send text to Ollama and return predicted sentiment label."""
    prompt = f"""Classify sentiment as Positive, Negative, or Neutral.
Only return ONE word.

User query: {user_query}
Sentiment:"""

    r = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "gemma3:1b",
            "prompt": prompt,
            "stream": False
        }
    )

    output = r.json()["response"].strip().lower()

    if "positive" in output:
        return "Positive"
    elif "negative" in output:
        return "Negative"
    else:
        return "Neutral"

In [4]:
# 
def qna_orchestrator(user_query, sentiment):
    """Send text to Ollama and return questions and answers depending on the sentiment"""

    if sentiment == "Positive":
        system_persona = "The user is happy, be friendly with him, try to use emojis and be enthusiastic."
    elif sentiment == "Negative":
        system_persona = "The user is unhappy, be careful and try to be supportive, helpful and give him a solution."
    elif sentiment == "Neutral":
        system_persona = "The user is neutral, provide clear and coloquial answers."

    prompt = f"""
### SYSTEM ROLE
You are a concise Q&A assistant. 

### INSTRUCTIONS
{system_persona}
- Keep your response brief and to the point.
- Match the tone specified in the persona above.

### USER QUERY
"{user_query}"

### ASSISTANT RESPONSE:
"""

    r = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "gemma3:1b",
            "prompt": prompt,
            "stream": False
        }
    )

    output = r.json()["response"].strip()

    return output

In [5]:
# Run Sentiment Analysis Evaluation
y_true = []
y_ollama = []

for text, label in test_data:
    y_true.append(label)

    y_ollama.append(sentiment_analysis(text))

print("\n=== OLLAMA ===")
print(classification_report(y_true, y_ollama))
print(confusion_matrix(y_true, y_ollama))


=== OLLAMA ===
              precision    recall  f1-score   support

    Negative       1.00      1.00      1.00         3
     Neutral       1.00      0.67      0.80         3
    Positive       0.80      1.00      0.89         4

    accuracy                           0.90        10
   macro avg       0.93      0.89      0.90        10
weighted avg       0.92      0.90      0.90        10

[[3 0 0]
 [0 2 1]
 [0 0 4]]


In [6]:
import gradio as gr

def master_process(user_query):
    # 1. First Step: Get Sentiment
    sentiment = sentiment_analysis(user_query)
    
    # 2. Second Step: Pass Sentiment into QnA Orchestrator
    # Note: I'm calling your qna_orchestrator exactly as you defined it
    ai_response = qna_orchestrator(user_query, sentiment)
    
    # 3. Return the sentiment (for a label) and the AI response
    return sentiment, ai_response

# --- UI Layout ---
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🤖 Sentiment-Aware AI Assistant")
    gr.Markdown("Type your question below. I'll detect your mood and adjust my response!")
    
    with gr.Row():
        with gr.Column(scale=2):
            input_box = gr.Textbox(label="User Query", placeholder="How is your day going?")
            submit_btn = gr.Button("Submit", variant="primary")
        
        with gr.Column(scale=1):
            sentiment_output = gr.Label(label="Detected Sentiment")

    response_output = gr.Textbox(label="AI Response", interactive=False)

    # --- Interaction Logic ---
    # When button is clicked, take input, and send outputs to the Label and Textbox
    submit_btn.click(
        fn=master_process, 
        inputs=input_box, 
        outputs=[sentiment_output, response_output]
    )

# Launch the app
demo.launch()

d:\Kike\Programacion\Language\lang\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
D:\_TEMPORALES_\USER\ipykernel_8720\2733397049.py:15: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
